# ACT-Break: Kumru Family Turkish Model Comparison

**Default profiles:** `kara-kumru` (`AlicanKiraz0/Kara-Kumru-v1.0-2B`) and `kumru` (`vngrs-ai/Kumru-2B`).

This notebook evaluates both Turkish Kumru-family models through the same ACT-Break pipeline without rewriting `config.py`. It uses the repository's model profile system, writes outputs to separate `outputs/<model-name>/` folders, and keeps activation success separate from behavioral text scoring.

### Google Colab Pro Recommendations
- **GPU Accelerator:** L4 or A100 is recommended. Step 1-4 are the core comparison path; Step 5-6 are much more expensive.
- **Google Drive Mount:** Recommended to preserve outputs, steering JSON files, figures, and validation results across runtime recycles.

## 1. Environment Setup & Dependency Installation

Mount Drive, clone or refresh the repo, install `uv`, and sync dependencies. If the repo already exists in the Colab runtime, this cell attempts a fast-forward pull so the notebook sees the latest profile and scoring code.

In [ ]:
from google.colab import drive
import os
import shutil
import subprocess
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Clone or refresh repository
repo_dir = Path('/content/ACT-Break')
if not repo_dir.exists():
    print('Cloning ACT-Break repository...')
    subprocess.run(['git', 'clone', 'https://github.com/IrohAmca/ACT-Break.git', str(repo_dir)], check=True)
else:
    print('ACT-Break repository already exists. Attempting fast-forward pull...')
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=False)

os.chdir(repo_dir)
print('Working directory:', Path.cwd())

# 3. Install uv package manager
if not shutil.which('uv'):
    print('Installing uv package manager...')
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + os.pathsep + os.environ['PATH']
else:
    print('uv is already installed.')

# 4. Sync dependencies
print('Syncing project dependencies via uv...')
subprocess.run(['uv', 'sync'], check=True)

# 5. Runtime defaults
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
print('Environment setup complete!')

## 2. GPU & Hardware Verification

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU Memory: {total_memory:.1f} GB')
else:
    print('WARNING: No GPU available. Runtime will be too slow for the full pipeline.')

## Hugging Face Authentication

Add `HF_TOKEN` in Colab secrets if a model or dependency requires authentication.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('Successfully logged in to Hugging Face!')
    else:
        print('HF_TOKEN was not set in Colab secrets; continuing without login.')
except Exception as exc:
    print(f'Failed to login to Hugging Face. Error: {exc}')

## 3. Kumru Comparison Runtime Configuration

Edit `MODEL_PROFILES_TO_RUN` if you only want one model. This cell preserves Colab environment variables you set manually, and only fills sensible defaults when they are missing.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

# Main comparison set. Edit this list if you want to run only one model.
MODEL_PROFILES_TO_RUN = ['kara-kumru', 'kumru']

# Core comparison path is Step 1-4. Step 5-6 are optional because GCG is expensive.
RUN_GCG_AND_MULTI_STAGE = False

# Defaults. If you set these in Colab before running the cell, your values are preserved.
os.environ.setdefault('ACT_BREAK_ADVBENCH_LANGUAGE', 'tr')
os.environ.setdefault('ACT_BREAK_STEERING_NUM_PROMPTS', '15')
os.environ.setdefault('ACT_BREAK_MAX_NEW_TOKENS', '20')

# Kumru-family profiles already default to this focused alpha range in config.py.
os.environ.setdefault('ACT_BREAK_STEERING_ALPHAS', '0,0.25,0.5,1,1.5,2,3,4,5')

MODEL_OUTPUT_SUFFIXES = {
    'kara-kumru': 'Kara-Kumru-v1.0-2B',
    'kumru': 'Kumru-2B',
    'kumru-base': 'Kumru-2B-Base',
}


def env_for_profile(profile: str) -> dict:
    env = os.environ.copy()
    env['ACT_BREAK_MODEL_PROFILE'] = profile
    env.setdefault('ACT_BREAK_ADVBENCH_LANGUAGE', 'tr')
    return env


def run_profile_script(profile: str, script: str, *args: str):
    print('\n' + '=' * 80)
    print(f'[{profile}] uv run python -u {script} ' + ' '.join(args))
    print('=' * 80)
    subprocess.run(
        ['uv', 'run', 'python', '-u', script, *args],
        env=env_for_profile(profile),
        check=True,
    )


def steering_json_path(profile: str) -> Path:
    suffix = MODEL_OUTPUT_SUFFIXES.get(profile)
    if suffix is None:
        raise KeyError(f'Unknown profile output suffix for {profile!r}')
    return Path('outputs') / suffix / 'steering' / 'steering_results.json'


print('Profiles to run:', MODEL_PROFILES_TO_RUN)
print('Steering prompts:', os.environ.get('ACT_BREAK_STEERING_NUM_PROMPTS'))
print('Steering alphas:', os.environ.get('ACT_BREAK_STEERING_ALPHAS'))
print('Run optional GCG + multi-stage:', RUN_GCG_AND_MULTI_STAGE)

## 4. Model Profile Preflight

Runs lightweight config checks for each selected profile before downloading full model weights or starting long GPU jobs.

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/00_check_model_config.py')

## 5. Build Turkish AdvBench Dataset

Creates `data/harmful_prompts_tr.csv` if it is missing. The command is resumable. For faster smoke tests, set `TRANSLATION_LIMIT` to a small integer before running.

In [ ]:
from pathlib import Path

TRANSLATION_PRESET = 'quality'
TRANSLATION_LIMIT = None  # Example: 50 for a quick smoke dataset.
tr_path = Path('data/harmful_prompts_tr.csv')

if tr_path.exists():
    print(f'Turkish AdvBench dataset already exists: {tr_path}')
else:
    args = ['--preset', TRANSLATION_PRESET]
    if TRANSLATION_LIMIT is not None:
        args.extend(['--limit', str(TRANSLATION_LIMIT)])
    run_profile_script(MODEL_PROFILES_TO_RUN[0], 'data/translate_advbench_tr.py', *args)

## 6. Turkish Pipeline Audit

Checks Turkish dataset columns, prompt loading, config profile, and tokenizer compatibility for each selected model.

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/audit_turkish_pipeline.py')

## 7. Core Comparison Pipeline: Step 1-4

These cells run each step for all selected profiles. Outputs are separated by model suffix, for example `outputs/Kara-Kumru-v1.0-2B/` and `outputs/Kumru-2B/`.

### Step 1: Collect Contrastive Activations

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/01_collect_activations.py')

### Step 2: Train Linear Probes

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/02_train_probe.py')

### Step 3: Extract Direction & Visualize

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/03_extract_direction.py')

### Step 4: Activation Steering Validation With Behavioral Scoring

This writes `steering_results.json` with `activation_success_counts`, `behavior_summary`, `steering_summary`, and per-response `behavior_score` fields.

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    run_profile_script(profile, 'scripts/04_steering_validation.py')

### Step 4 Summary: Compare Steering Metrics

In [ ]:
for profile in MODEL_PROFILES_TO_RUN:
    path = steering_json_path(profile)
    print('\n' + '=' * 80)
    print(f'[{profile}] {path}')
    if not path.exists():
        print('Missing steering results. Run Step 4 first.')
        continue

    data = json.loads(path.read_text(encoding='utf-8'))
    print('Model:', data.get('model_info', {}).get('model_name'))
    print('Schema:', data.get('schema_version'))
    print('Total prompts:', data.get('total_prompts'))
    print('alpha | act | non_refusal | candidate | echo | repeat | labels')
    for alpha in data.get('alphas', []):
        key = str(alpha)
        row = data.get('steering_summary', {}).get(key, {})
        print(
            f"{key:>5} | "
            f"{row.get('activation_success_count', 0):>3} | "
            f"{row.get('behavioral_non_refusal', 0):>11} | "
            f"{row.get('behavioral_jailbreak_candidate', 0):>9} | "
            f"{row.get('prompt_echo', 0):>4} | "
            f"{row.get('repetition_collapse', 0):>6} | "
            f"{row.get('labels', {})}"
        )

## 8. Optional: GCG Suffix Optimization and Multi-Stage Validation

Leave `RUN_GCG_AND_MULTI_STAGE = False` for the first comparison pass. Flip it to `True` in the runtime configuration cell when you want to run the expensive suffix optimization path for both models.

In [ ]:
if RUN_GCG_AND_MULTI_STAGE:
    for profile in MODEL_PROFILES_TO_RUN:
        run_profile_script(profile, 'scripts/05_optimize_suffix.py')
        run_profile_script(profile, 'scripts/06_multi_stage_validation.py')
else:
    print('Skipped Step 5-6. Set RUN_GCG_AND_MULTI_STAGE = True in the runtime configuration cell to run them.')

## 9. Backup Results to Google Drive

Copies the whole `outputs/` tree plus key repo files to a timestamped Google Drive folder.

In [ ]:
import os
import shutil
from datetime import datetime, timezone

profiles_slug = '-'.join(MODEL_PROFILES_TO_RUN)
drive_dest = f'/content/drive/My Drive/ACT-Break-Results-{profiles_slug}'

if os.path.exists('/content/drive/My Drive'):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    run_dest = os.path.join(drive_dest, f'run_{timestamp}')
    os.makedirs(run_dest, exist_ok=True)

    print(f'Copying outputs to Google Drive: {run_dest}')
    if os.path.exists('outputs'):
        shutil.copytree('outputs', os.path.join(run_dest, 'outputs'), dirs_exist_ok=True)
        for file in ['README.md', 'config.py', 'colab/colab_notebook_karakumru.ipynb']:
            if os.path.exists(file):
                dest_file = os.path.join(run_dest, file.replace('/', '_'))
                shutil.copy(file, dest_file)
        print('Successfully copied outputs to Google Drive!')
    else:
        print('No outputs directory found yet. Please run the pipeline steps first.')
else:
    print('Google Drive is not mounted. Please mount Google Drive in the setup cell first.')